# TP-2: CleaningData - Applying OOP and Design Patterns
Each exercise builds upon the previous one - so by the end, we will have a complete CSV cleaning data using OOP, decorators, and patterns.



## Exercise 1: Build the CSVReader Class (OOP Foundation)

### Objective:
•	Reinforce OOP fundamentals: classes, constructors, methods, encapsulation.

•	Learn to design reusable data classes for DS projects.
### Concept:
A CSVReader class should encapsulate all functionality related to reading CSV files.
### Instructions:
1.	Create a class CSVReader.
2.	Define an __init__ method that takes file_path.
3.	Add a read() method that loads the CSV file using pandas.
4.	Add a preview(n) method that prints the top n rows.



In [128]:
import os
import pandas as pd
import numpy as np 
from abc import ABC, abstractmethod
from typing import Protocol
import time 
import functools
from pathlib import Path

In [129]:
class CSVReader:
    

    def __init__(self, file_path):
        self.file_path = file_path
        self._data = None  

    def read(self):
        
        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f"File not found: {self.file_path}")
        self._data = pd.read_csv(self.file_path)
        return self._data

    def preview(self, n=5):
        
        if self._data is None:
            self.read()
        print(f"Top {n} rows of '{self.file_path}':")
        print(self._data.head(n))

    @property
    def data(self):
       
        return self._data

    def __repr__(self):
        status = "loaded" if self._data is not None else "not loaded"
        return f"CSVReader(file_path='{self.file_path}', status={status})"

In [130]:
reader = CSVReader("sample_data (2).csv")
print(reader)

CSVReader(file_path='sample_data (2).csv', status=not loaded)


In [131]:
df = reader.read()
print(reader)
print("Shape:", df.shape)
df

CSVReader(file_path='sample_data (2).csv', status=loaded)
Shape: (7, 7)


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


In [132]:
reader.preview(3)

Top 3 rows of 'sample_data (2).csv':
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN


In [133]:
print(type(reader.data))
print("Missing values per column:")
print(reader.data.isnull().sum())

<class 'pandas.DataFrame'>
Missing values per column:
id           0
name         0
age          3
height_cm    1
weight_kg    1
city         0
score        2
dtype: int64


In [134]:
try:
    CSVReader("not_exist.csv").read()
except FileNotFoundError as e:
    print("Error:", e)

Error: File not found: not_exist.csv


## Exercise 2: Apply Pattern for Missing Values Handling

### Objective:
•	Learn the Strategy Pattern - one interface, many interchangeable behaviors.

•	Practice inheritance and polymorphism.
### Concept:
We often need different strategies to handle missing data (drop, fill mean, fill mode, etc.).
The pattern allows flexible switching between methods without changing the main code.
### Instructions:
1.	Create an abstract class MissingValueStrategy with a method handle(df).
2.	Create subclasses:
o	DropMissing
o	FillMean
o	FillMode
3.	Create a DataCleaner class that applies the chosen strategy.


In [135]:
df = pd.read_csv('sample_data (2).csv')

print("Original Dataset:")
print(df)
print("\nMissing Values Count:")
print(df.isnull().sum())
print("\nData Types:")
print(df.dtypes)

Original Dataset:
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0
5   6    Frank  30.0      175.0       85.0  Los Angeles    NaN
6   7    Grace   NaN      162.0       60.0      Chicago   92.0

Missing Values Count:
id           0
name         0
age          3
height_cm    1
weight_kg    1
city         0
score        2
dtype: int64

Data Types:
id             int64
name             str
age          float64
height_cm    float64
weight_kg    float64
city             str
score        float64
dtype: object


In [136]:
class MissingValueStrategy(ABC):
    """Abstract base class for missing value handling strategies."""
    
    @abstractmethod
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """Handle missing values in the dataframe.
        
        Args:
            df: Input dataframe with missing values
            
        Returns:
            DataFrame with missing values handled
        """
        pass
    
    def __str__(self):
        return self.__class__.__name__

In [137]:
class DropMissing(MissingValueStrategy):
    """Strategy to drop rows with any missing values."""
    
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """Drop all rows containing missing values."""
        result = df.dropna()
        print(f"Dropped {len(df) - len(result)} rows with missing values.")
        return result

In [138]:
class FillMean(MissingValueStrategy):
    """Strategy to fill missing values with the mean of numeric columns."""
    
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """Fill missing values with column means for numeric columns."""
        result = df.copy()
        
        # Get numeric columns only
        numeric_cols = result.select_dtypes(include=[np.number]).columns
        
        for col in numeric_cols:
            if result[col].isnull().any():
                mean_value = result[col].mean()
                result[col].fillna(mean_value, inplace=True)
                print(f"Filled {col} with mean: {mean_value:.2f}")
        
        # For non-numeric columns, we could drop rows or use a different strategy
        # Here we'll just leave them as is
        return result

In [139]:
class FillMode(MissingValueStrategy):
    """Strategy to fill missing values with the mode (most frequent value)."""
    
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """Fill missing values with column mode."""
        result = df.copy()
        
        for col in result.columns:
            if result[col].isnull().any():
                mode_value = result[col].mode()
                if len(mode_value) > 0:
                    result[col].fillna(mode_value[0], inplace=True)
                    print(f"Filled {col} with mode: {mode_value[0]}")
        
        return result

In [140]:
class DataCleaner:
    """Context class that applies a missing value handling strategy."""
    
    def __init__(self, strategy: MissingValueStrategy):
        """Initialize with a specific strategy.
        
        Args:
            strategy: A MissingValueStrategy instance
        """
        self._strategy = strategy
    
    def set_strategy(self, strategy: MissingValueStrategy):
        """Change the strategy at runtime.
        
        Args:
            strategy: A new MissingValueStrategy instance
        """
        self._strategy = strategy
        print(f"Strategy changed to: {strategy}")
    
    def clean_data(self, df: pd.DataFrame) -> pd.DataFrame:
        """Clean the dataframe using the current strategy.
        
        Args:
            df: Input dataframe
            
        Returns:
            Cleaned dataframe
        """
        print(f"\nApplying strategy: {self._strategy}")
        print("-" * 50)
        return self._strategy.handle(df)

In [141]:
# Create a cleaner with DropMissing strategy
cleaner = DataCleaner(DropMissing())
df_dropped = cleaner.clean_data(df)

print("\nResult after dropping missing values:")
print(df_dropped)
print(f"\nShape: {df_dropped.shape}")


Applying strategy: DropMissing
--------------------------------------------------
Dropped 5 rows with missing values.

Result after dropping missing values:
   id   name   age  height_cm  weight_kg      city  score
0   1  Alice  29.0      165.0       68.0  New York   85.0
4   5    Eva  27.0      160.0       55.0  New York   88.0

Shape: (2, 7)


In [142]:
# Change strategy to FillMean
cleaner.set_strategy(FillMean())
df_mean_filled = cleaner.clean_data(df)

print("\nResult after filling with mean:")
print(df_mean_filled)
print("\nMissing values after filling with mean:")
print(df_mean_filled.isnull().sum())

Strategy changed to: FillMean

Applying strategy: FillMean
--------------------------------------------------
Filled age with mean: 30.25
Filled height_cm with mean: 167.00
Filled weight_kg with mean: 70.00
Filled score with mean: 86.00

Result after filling with mean:
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0
5   6    Frank  30.0      175.0       85.0  Los Angeles    NaN
6   7    Grace   NaN      162.0       60.0      Chicago   92.0

Missing values after filling with mean:
id           0
name         0
age          3
height_cm    1
weight_kg    1
city         0
score        2
dtype: int64


C:\Users\Sour Aliya\AppData\Local\Temp\ipykernel_21168\940559035.py:14: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  result[col].fillna(mean_value, inplace=True)


In [143]:
# Change strategy to FillMode
cleaner.set_strategy(FillMode())
df_mode_filled = cleaner.clean_data(df)

print("\nResult after filling with mode:")
print(df_mode_filled)
print("\nMissing values after filling with mode:")
print(df_mode_filled.isnull().sum())

Strategy changed to: FillMode

Applying strategy: FillMode
--------------------------------------------------
Filled age with mode: 27.0
Filled height_cm with mode: 160.0
Filled weight_kg with mode: 55.0
Filled score with mode: 75.0

Result after filling with mode:
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0
5   6    Frank  30.0      175.0       85.0  Los Angeles    NaN
6   7    Grace   NaN      162.0       60.0      Chicago   92.0

Missing values after filling with mode:
id           0
name         0
age          3
height_cm    1
weight_kg    1
city         0
score        2
dtype: int64


C:\Users\Sour Aliya\AppData\Local\Temp\ipykernel_21168\4106858363.py:12: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  result[col].fillna(mode_value[0], inplace=True)


In [144]:
print("=" * 60)
print("COMPARISON OF STRATEGIES")
print("=" * 60)

print(f"\nOriginal Dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Missing values: {df.isnull().sum().sum()}")

print(f"\n1. DropMissing: {df_dropped.shape[0]} rows remaining")
print(f"   Data loss: {((df.shape[0] - df_dropped.shape[0]) / df.shape[0] * 100):.1f}%")

print(f"\n2. FillMean: {df_mean_filled.shape[0]} rows, {df_mean_filled.isnull().sum().sum()} missing values")
print(f"   Numeric columns filled with mean")

print(f"\n3. FillMode: {df_mode_filled.shape[0]} rows, {df_mode_filled.isnull().sum().sum()} missing values")
print(f"   All columns filled with mode (most frequent value)")

COMPARISON OF STRATEGIES

Original Dataset: 7 rows, 7 columns
Missing values: 7

1. DropMissing: 2 rows remaining
   Data loss: 71.4%

2. FillMean: 7 rows, 7 missing values
   Numeric columns filled with mean

3. FillMode: 7 rows, 7 missing values
   All columns filled with mode (most frequent value)


In [145]:
class FillMedian(MissingValueStrategy):
    """Strategy to fill missing values with the median of numeric columns."""
    
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """Fill missing values with column medians for numeric columns."""
        result = df.copy()
        
        numeric_cols = result.select_dtypes(include=[np.number]).columns
        
        for col in numeric_cols:
            if result[col].isnull().any():
                median_value = result[col].median()
                result[col].fillna(median_value, inplace=True)
                print(f"Filled {col} with median: {median_value:.2f}")
        
        return result

# Test the custom strategy
cleaner.set_strategy(FillMedian())
df_median_filled = cleaner.clean_data(df)

print("\nResult after filling with median:")
print(df_median_filled)

Strategy changed to: FillMedian

Applying strategy: FillMedian
--------------------------------------------------
Filled age with median: 29.50
Filled height_cm with median: 166.50
Filled weight_kg with median: 70.00
Filled score with median: 88.00

Result after filling with median:
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0
5   6    Frank  30.0      175.0       85.0  Los Angeles    NaN
6   7    Grace   NaN      162.0       60.0      Chicago   92.0


C:\Users\Sour Aliya\AppData\Local\Temp\ipykernel_21168\2426729673.py:13: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  result[col].fillna(median_value, inplace=True)
C:\Users\Sour Aliya\AppData\Local\Temp\ipykernel_21168\2426729673.py:13: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained 

## Exercise 3: Add Decorators for Logging and Timing
### Objective:
•	Learn the Decorator Pattern to add reusable functionality (logging, timing).

•	Understand how decorators support the “Open/Closed Principle”.
### Concept:
Decorators wrap functions to extend their behavior without altering the original code.
### Instructions:
1.	Create two decorators:
o	@log_action → logs when a method starts and finishes.
o	@log_time → calculates and prints execution time.
2.	Apply them to methods in CSVReader or DataCleaner.


In [146]:
# 1. Decorator for logging
def log_action(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[START] {func.__name__}")
        result = func(*args, **kwargs)
        print(f"[FINISH] {func.__name__}")
        return result
    return wrapper

In [147]:
# 2. Decorator for execution time
def log_time(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        end = time.perf_counter()

        print(f"[TIME] {func.__name__}: {end - start:.6f} seconds")
        return result
    return wrapper

In [148]:
# 3. CSVReader class from Exercise 1
class CSVReader:
    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None

    @log_action
    @log_time
    def read(self):
        self.df = pd.read_csv(self.file_path)
        return self.df

    @log_action
    def preview(self, n=5):
        if self.df is None:
            self.read()
        print(self.df.head(n))

In [149]:
# 4. Test with sample CSV
reader = CSVReader("sample_data (2).csv")

df = reader.read()
reader.preview(5)

[START] read
[TIME] read: 0.001920 seconds
[FINISH] read
[START] preview
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0
[FINISH] preview


## Exercise 4: Implement Factory Pattern for Data Transformations
### Objective:
•	Practice Factory Pattern for scalable creation of transformation objects.

•	Apply abstraction and composition.
### Concept:
Instead of manually creating objects, use a factory that decides what transformation to apply.
### Instructions:
1.	Create an abstract class DataTransform with a method apply(df).
2.	Implement subclasses:
o	NormalizeColumns
o	RemoveDuplicates
o	StandardizeText
3.	Create a TransformFactory that returns transformation objects based on string input.


In [150]:
# 1. Abstract Class
class DataTransform(ABC):
    @abstractmethod
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        pass

In [151]:
class NormalizeColumns(DataTransform):
    """Makes column names lowercase and replaces spaces with underscores."""
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df.columns = [col.strip().lower().replace(" ", "_") for col in df.columns]
        return df

In [152]:
class RemoveDuplicates(DataTransform):
    """Removes duplicate rows from the DataFrame."""
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.drop_duplicates().reset_index(drop=True)

In [153]:
class StandardizeText(DataTransform):
    """Trims extra spaces and converts text values to lowercase."""
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        text_cols = df.select_dtypes(include=["object", "string"]).columns
        for col in text_cols:
            df[col] = df[col].astype(str).str.strip().str.lower()
        return df

In [154]:
# 3. Factory Class
class TransformFactory:
    @staticmethod
    def create_transform(transform_type: str) -> DataTransform:
        transforms = {
            "normalize_columns": NormalizeColumns,
            "remove_duplicates": RemoveDuplicates,
            "standardize_text": StandardizeText,
        }
        key = transform_type.strip().lower()
        if key not in transforms:
            raise ValueError(f"Unknown transformation: '{transform_type}'")
        return transforms[key]()

In [155]:
df = pd.read_csv("sample_data (2).csv")

# Apply transformations using string names
pipeline = ["normalize_columns", "standardize_text", "remove_duplicates"]

for step in pipeline:
    transformer = TransformFactory.create_transform(step)
    df = transformer.apply(df)

print(df)

   id     name   age  height_cm  weight_kg         city  score
0   1    alice  29.0      165.0       68.0     new york   85.0
1   2      bob   NaN      172.0        NaN  los angeles   90.0
2   3  charlie  35.0      168.0       72.0      chicago    NaN
3   4    david   NaN        NaN       80.0      houston   75.0
4   5      eva  27.0      160.0       55.0     new york   88.0
5   6    frank  30.0      175.0       85.0  los angeles    NaN
6   7    grace   NaN      162.0       60.0      chicago   92.0


## Exercise 5: Build a Full Cleaning Pipeline (Template Method Pattern)
### Objective:
•	Combine all previous concepts into one cleaning pipeline.

•	Use the Template Method Pattern to define a consistent workflow.
### Concept:
The Template Method defines the skeleton of a process and lets subclasses override specific steps.
### Instructions:
1.	Create an abstract class DataPipeline with a run() method defining steps:
o	load()
o	clean()
o	transform()
o	save()
2.	Create CSVDataPipeline that implements each step using previous classes.


In [156]:


def log_action(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Starting {func.__name__}...")
        try:
            return func(*args, **kwargs)
        finally:
            print(f"Finished {func.__name__}.")

    return wrapper


def log_time(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            print(f"{func.__name__} took {time.perf_counter() - start:.4f} seconds.")

    return wrapper


class CSVReader:
    def __init__(self, file_path: str) -> None:
        self.file_path = Path(file_path)
        self.df: pd.DataFrame | None = None

    @log_action
    @log_time
    def read(self) -> pd.DataFrame:
        self.df = pd.read_csv(self.file_path)
        return self.df

    def preview(self, n: int = 5) -> None:
        if self.df is None:
            raise RuntimeError("Read the CSV before previewing it.")
        print(self.df.head(n))

In [157]:
class MissingValueStrategy(ABC):
    @abstractmethod
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """Handle missing values and return a dataframe."""


class DropMissing(MissingValueStrategy):
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.dropna().copy()


class FillMean(MissingValueStrategy):
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        result = df.copy()
        numeric_columns = result.select_dtypes(include="number").columns
        result[numeric_columns] = result[numeric_columns].fillna(
            result[numeric_columns].mean()
        )
        return result


class FillMode(MissingValueStrategy):
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        result = df.copy()
        for column in result.columns:
            modes = result[column].mode(dropna=True)
            if not modes.empty:
                result[column] = result[column].fillna(modes.iloc[0])
        return result


class DataCleaner:
    def __init__(self, strategy: MissingValueStrategy) -> None:
        self.strategy = strategy

    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        return self.strategy.handle(df)


class DataPipeline(ABC):
    """Abstract base class defining the Template Method workflow."""

    def run(self) -> None:
        """Run the pipeline steps in a fixed order."""
        self.load()
        self.clean()
        self.transform()
        self.save()

    @abstractmethod
    def load(self) -> None:
        """Load raw data into the pipeline."""

    @abstractmethod
    def clean(self) -> None:
        """Handle missing values and duplicate records."""

    @abstractmethod
    def transform(self) -> None:
        """Apply transformations to the cleaned data."""

    @abstractmethod
    def save(self) -> None:
        """Save the processed data."""



In [158]:
class DataTransform(ABC):
    @abstractmethod
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        """Apply a transformation and return the resulting dataframe."""


class NormalizeColumns(DataTransform):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        result = df.copy()
        numeric_columns = result.select_dtypes(include="number").columns
        for column in numeric_columns:
            minimum = result[column].min()
            maximum = result[column].max()
            if maximum != minimum:
                result[column] = (result[column] - minimum) / (maximum - minimum)
        return result


class RemoveDuplicates(DataTransform):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.drop_duplicates().copy()


class StandardizeText(DataTransform):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        result = df.copy()
        text_columns = result.select_dtypes(include=["object", "string"]).columns
        for column in text_columns:
            result[column] = result[column].str.strip().str.title()
        return result


class TransformFactory:
    @staticmethod
    def create(name: str) -> DataTransform:
        transformations = {
            "normalize": NormalizeColumns,
            "remove_duplicates": RemoveDuplicates,
            "standardize_text": StandardizeText,
        }
        try:
            transformation = transformations[name.lower()]
        except KeyError as error:
            raise ValueError(f"Unknown transformation: {name}") from error
        return transformation()


class CSVDataPipeline(DataPipeline):
    """CSV pipeline that composes the reader, cleaner, and transform factory."""

    def __init__(self, input_filepath: str, output_filepath: str) -> None:
        self.reader = CSVReader(input_filepath)
        self.output_filepath = Path(output_filepath)
        self.cleaner = DataCleaner(FillMean())
        self.transformations = [
            TransformFactory.create("remove_duplicates"),
            TransformFactory.create("standardize_text"),
        ]
        self.df: pd.DataFrame | None = None

    def load(self) -> None:
        self.df = self.reader.read()
        print(f"Loaded {len(self.df)} rows from {self.reader.file_path}.")

    def clean(self) -> None:
        if self.df is None:
            raise RuntimeError("Load the data before cleaning it.")
        self.df = self.cleaner.clean(self.df)
        print("Filled missing numeric values using column means.")

    def transform(self) -> None:
        if self.df is None:
            raise RuntimeError("Load the data before transforming it.")
        for transformation in self.transformations:
            self.df = transformation.apply(self.df)
        print("Removed duplicates and standardized text values.")

    def save(self) -> None:
        if self.df is None:
            raise RuntimeError("Load the data before saving it.")
        self.output_filepath.parent.mkdir(parents=True, exist_ok=True)
        self.df.to_csv(self.output_filepath, index=False)
        print(f"Saved cleaned data to {self.output_filepath}.")




In [159]:
# Run all pipeline steps using the supplied CSV file.
input_file = Path("sample_data (2).csv")
if not input_file.exists():
    input_file = Path("ADV") / input_file
pipeline = CSVDataPipeline(str(input_file), "cleaned_sample_data.csv")
pipeline.run()
pipeline.reader.preview(5)

Starting read...
read took 0.0027 seconds.
Finished read.
Loaded 7 rows from sample_data (2).csv.
Filled missing numeric values using column means.
Removed duplicates and standardized text values.
Saved cleaned data to cleaned_sample_data.csv.
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0
